In [ ]:
# dataset_prep.py

import os
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from data_utils import (
    get_midi_number,
    save_pickle,
    normalize_spelled_pitch,        # 导入标准化函数
    ENHARMONIC_MAPPING,             # 导入映射字典
    REVERSE_ENHARMONIC_MAPPING,     # 导入反向映射字典
    CustomFingeringEncoder
)


def mark_chords(df):
    """
    标记 'chord' 列为 1 如果同一曲目 (piece_id) 同一手部 (hand) 内有多个音符的时间区间重叠。
    否则，标记为 0。
    """
    df['chord'] = 0  # 初始化为 0

    # 按 'piece_id' 和 'hand' 分组
    grouped = df.groupby(['piece_id', 'hand'])

    def mark_chords_group(group):
        """
        对每个分组（同一曲目同一手部）标记和弦。
        """
        group = group.sort_values('onset_time').reset_index(drop=True)
        n = len(group)
        chord_indices = set()

        for i in range(n):
            current_onset = group.loc[i, 'onset_time']
            current_offset = group.loc[i, 'offset_time']

            for j in range(i + 1, n):
                next_onset = group.loc[j, 'onset_time']
                next_offset = group.loc[j, 'offset_time']

                if next_onset < current_offset:  # 重叠
                    chord_indices.add(i)
                    chord_indices.add(j)
                else:
                    break  # 因为已排序，后续不会再重叠

        # 将重叠的音符标记为和弦
        if chord_indices:
            group.loc[list(chord_indices), 'chord'] = 1

        return group

    # 应用到每个分组
    df = grouped.apply(mark_chords_group).reset_index(drop=True)

    return df

def parse_fingering_file(file_path):
    """
    解析单个 fingering 文件，返回包含所有音符信息的列表。
    """
    piece_id = os.path.splitext(os.path.basename(file_path))[0]  # 使用文件名（不含扩展名）作为 piece_id
    data = []
    with open(file_path, 'r') as f:
        for line in f:
            # 去除首尾空白字符并按空格分割
            parts = line.strip().split()
            if len(parts) < 8:
                continue  # 跳过格式不完整的行
            note_id = parts[0]
            onset_time = float(parts[1])
            offset_time = float(parts[2])
            spelled_pitch = parts[3]
            onset_velocity = float(parts[4])
            offset_velocity = float(parts[5])
            channel = int(parts[6])
            finger_number = parts[7]

            # 处理指法替换（例如 '3_1'）
            if '_' in finger_number:
                finger_number = finger_number.split('_')[0]  # 仅取主要指法

            # 转换指法为整数
            try:
                finger_number = int(finger_number)
            except ValueError:
                finger_number = None  # 处理无法转换的指法

            # 解析音高和八度
            pitch_name = ''.join([c for c in spelled_pitch if c.isalpha() or c in ['#', 'b']])
            octave = ''.join([c for c in spelled_pitch if c.isdigit()])
            octave = int(octave) if octave else 4  # 默认八度为4

            # 确定手部
            hand = 'right' if channel == 0 else 'left'

            # 计算音符持续时间，并统一小数位数
            duration = round(offset_time - onset_time, 2)

            # 标准化 spelled_pitch
            normalized_spelled_pitch = normalize_spelled_pitch(spelled_pitch)

            # 统一小数位数的处理
            onset_time = round(onset_time, 3)
            offset_time = round(offset_time, 3)

            data.append({
                'piece_id': piece_id,  # 添加 piece_id
                'note_id': note_id,
                'onset_time': onset_time,
                'offset_time': offset_time,
                'spelled_pitch': spelled_pitch,
                'normalized_spelled_pitch': normalized_spelled_pitch,  # 添加标准化后的音符
                'pitch_name': pitch_name,
                'octave': octave,
                'duration': duration,
                'hand': hand,
                'finger_number': finger_number
            })
    return data

def load_pig_dataset(fingering_dir):
    """
    加载 PIG Dataset 中所有 fingering 文件，返回一个包含所有音符数据的 DataFrame。
    """
    all_data = []
    for file_name in os.listdir(fingering_dir):
        if file_name.endswith('.txt'):
            file_path = os.path.join(fingering_dir, file_name)
            piece_data = parse_fingering_file(file_path)
            all_data.extend(piece_data)
    df = pd.DataFrame(all_data)
    return df

def plot_duration_distribution(df):
    """
    绘制音符持续时间的分布图。
    """
    plt.figure(figsize=(10, 6))
    sns.histplot(df['duration'], bins=50, kde=True, color='skyblue')
    plt.title('Distribution of Note Durations')
    plt.xlabel('Duration (s)')
    plt.ylabel('Frequency')
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('duration_distribution.png')
    plt.show()

def main():
    # 设置 fingering 文件夹路径
    fingering_folder = 'PIGdata/FingeringFiles'  # 请根据实际路径调整

    # 加载数据
    df = load_pig_dataset(fingering_folder)

    # 查看数据
    print("初始数据样例：")
    print(df.head())

    # 删除指法缺失的音符
    df = df.dropna(subset=['finger_number'])

    # 确保指法为整数类型
    df['finger_number'] = df['finger_number'].astype(int)

    # 统一小数位数
    float_columns = ['onset_time', 'offset_time', 'duration']
    df[float_columns] = df[float_columns].round(2)

    # 计算 MIDI 编号，使用标准化后的音符
    df['midi_number'] = df['normalized_spelled_pitch'].apply(get_midi_number)

    # 初始化 LabelEncoder
    le_pitch = LabelEncoder()
    le_duration = LabelEncoder()
    le_hand = LabelEncoder()

    # 对类别特征进行标签编码，使用标准化后的音符
    df['pitch_encoded'] = le_pitch.fit_transform(df['normalized_spelled_pitch'])
    df['duration_encoded'] = le_duration.fit_transform(df['duration'].astype(str))
    df['hand_encoded'] = le_hand.fit_transform(df['hand'])

    fingering_encoder = CustomFingeringEncoder()
    print("Original finger numbers:", np.unique(df['finger_number']))

    # 确保所有指法都是合法的
    fingering_encoder.fit(df['finger_number'])
    df['fingering_encoded'] = fingering_encoder.transform(df['finger_number'])

    print("Encoded finger numbers:", np.unique(df['fingering_encoded']))
    print("\nFingering mapping:")
    for orig, encoded in fingering_encoder.mapping.items():
        print(f"{orig} -> {encoded}")

    df = mark_chords(df)

    # 特征和标签
    X = df[['pitch_encoded', 'duration_encoded', 'hand_encoded']].values
    y = df['fingering_encoded'].values

    print(f"特征形状: {X.shape}")
    print(f"标签形状: {y.shape}")
    # 保存 LabelEncoder 和映射
    save_pickle(le_pitch, 'le_pitch.pkl')
    save_pickle(le_duration, 'le_duration.pkl')
    save_pickle(le_hand, 'le_hand.pkl')
    save_pickle(fingering_encoder, 'le_fingering.pkl')
    save_pickle(df, "df.pkl")
    # 保存标准化映射
    save_pickle(ENHARMONIC_MAPPING, 'enharmonic_mapping.pkl')
    save_pickle(REVERSE_ENHARMONIC_MAPPING, 'reverse_enharmonic_mapping.pkl')

    # 绘制 duration 分布图
    plot_duration_distribution(df)

    # 创建序列
    sequence_length = 10  # 使用前10个音符预测第11个音符

    def create_sequences(X, y, seq_length):
        X_seq = []
        y_seq = []
        for i in range(len(X) - seq_length):
            X_seq.append(X[i:i + seq_length])
            y_seq.append(y[i + seq_length])
        return np.array(X_seq), np.array(y_seq)

    X_seq, y_seq = create_sequences(X, y, sequence_length)

    print(f"序列特征形状: {X_seq.shape}")  # (样本数, sequence_length, 特征数量)
    print(f"序列标签形状: {y_seq.shape}")  # (样本数,)

    # 查看不同时长的分布
    duration_counts = df['duration'].value_counts().sort_index()
    print("\n不同时长的分布：")
    print(duration_counts)

    # 将序列数据保存为 npy 文件
    np.save('X_train.npy', X_seq)
    np.save('X_val.npy', X_seq)  # 这里需要根据实际划分修改
    np.save('y_train.npy', y_seq)
    np.save('y_val.npy', y_seq)  # 这里需要根据实际划分修改

    print("序列数据已保存。")

if __name__ == "__main__":
    main()


In [ ]:
# data_process.py

import numpy as np
import pandas as pd
import pickle
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import sklearn_crfsuite  # Add CRF import
from sklearn_crfsuite import metrics
from data_utils import (
    get_midi_number,
    is_black_key,
    calculate_speed_features,
    calculate_midi_diff,
    create_word_column,
    train_word2vec,
    get_fused_features,
    combine_features,
    save_pickle,
    load_pickle
)
from gensim.models import Word2Vec


# Function to convert sequences to CRF features
def sequence_to_crf_features(sequence, window_size=2):
    """
    Convert a sequence of notes to CRF features with context window.

    Args:
        sequence: A sequence of feature dictionaries
        window_size: Number of notes to consider before and after

    Returns:
        List of dictionaries with features for CRF
    """
    features = []
    seq_len = len(sequence)

    for i in range(seq_len):
        # Basic features for current note
        note_features = {
            'pitch': sequence[i]['pitch_encoded'],
            'duration': sequence[i]['duration_encoded'],
            'hand': sequence[i]['hand_encoded'],
            'black_key': sequence[i]['black_key'],
            'chord': sequence[i]['chord'],
            'midi_diff': sequence[i]['midi_diff_processed'],
            'density': sequence[i]['note_density'],
        }

        # Add context features (previous and next notes)
        for offset in range(-window_size, window_size + 1):
            if offset == 0:  # Skip current note (already added)
                continue

            idx = i + offset
            # Handle boundary conditions
            if 0 <= idx < seq_len:
                prefix = 'prev' if offset < 0 else 'next'
                abs_offset = abs(offset)
                note_features.update({
                    f'{prefix}{abs_offset}_pitch': sequence[idx]['pitch_encoded'],
                    f'{prefix}{abs_offset}_duration': sequence[idx]['duration_encoded'],
                    f'{prefix}{abs_offset}_hand': sequence[idx]['hand_encoded'],
                    f'{prefix}{abs_offset}_black_key': sequence[idx]['black_key'],
                    f'{prefix}{abs_offset}_chord': sequence[idx]['chord'],
                })
        else:
                # For out of bounds, use special indicators
                prefix = 'prev' if offset < 0 else 'next'
                abs_offset = abs(offset)
                note_features.update({
                    f'{prefix}{abs_offset}_pitch': -1,
                    f'{prefix}{abs_offset}_duration': -1,
                    f'{prefix}{abs_offset}_hand': -1,
                    f'{prefix}{abs_offset}_black_key': -1,
                    f'{prefix}{abs_offset}_chord': -1,
                })

        features.append(note_features)

    return features


# Function to train CRF model and extract CRF features
def extract_crf_features(df, sequence_length=10):
    """
    Train a CRF model and extract CRF features

    Args:
        df: DataFrame with note data
        sequence_length: Length of sequences to consider

    Returns:
        DataFrame with added CRF features
    """
    print("Training CRF model and extracting CRF features...")

    # Convert DataFrame rows to dictionaries for CRF feature extraction
    df_dict = df.to_dict('records')

    # Extract CRF features for each sequence
    crf_features = []
    for i in range(0, len(df_dict), sequence_length):
        if i + sequence_length <= len(df_dict):
            seq = df_dict[i:i+sequence_length]
            crf_features.extend(sequence_to_crf_features(seq))

    # Ensure the feature list is the same length as the DataFrame
    if len(crf_features) < len(df):
        # Pad with empty features for any remaining rows
        remainder = len(df) - len(crf_features)
        empty_features = [{'crf_placeholder': 0} for _ in range(remainder)]
        crf_features.extend(empty_features)
    elif len(crf_features) > len(df):
        # Trim excess features
        crf_features = crf_features[:len(df)]

    # Train CRF model on sequences and extract probabilities
    X_crf = []
    y_crf = []

    for i in range(0, len(df) - sequence_length):
        X_crf.append(crf_features[i:i+sequence_length])
        y_crf.append([str(y) for y in df['fingering_encoded'].values[i:i+sequence_length]])

    # Train CRF model
    crf = sklearn_crfsuite.CRF(
        algorithm='lbfgs',
        c1=0.1,
        c2=0.1,
        max_iterations=100,
        all_possible_transitions=True
    )

    if len(X_crf) > 0:
        print(f"Training CRF model with {len(X_crf)} sequences...")
        crf.fit(X_crf, y_crf)

        # Extract probability features for each position
        crf_probs = []
        for seq_features in X_crf:
            seq_probs = crf.predict_marginals_single(seq_features)
            crf_probs.extend(seq_probs)

        # Flatten CRF probabilities to a fixed-size vector for each note
        max_classes = max(len(probs) for probs in crf_probs) if crf_probs else 0
        crf_vectors = []

        for probs in crf_probs:
            vector = []
            for i in range(max_classes):
                class_key = str(i)
                vector.append(probs.get(class_key, 0.0))
            crf_vectors.append(vector)

        # Pad with zeros for notes without CRF features
        zero_vector = [0.0] * max_classes
        while len(crf_vectors) < len(df):
            crf_vectors.append(zero_vector)

        # Add CRF vectors to DataFrame
        df['crf_feature'] = crf_vectors[:len(df)]
    else:
        print("WARNING: Not enough data to train CRF model")
        # Add empty CRF features
        df['crf_feature'] = [[0.0]] * len(df)

    return df


def main():
    # 加载序列数据
    try:
        X_train = np.load('X_train.npy')
        X_val = np.load('X_val.npy')
        y_train = np.load('y_train.npy')
        y_val = np.load('y_val.npy')
    except FileNotFoundError as e:
        print(f"Error loading data files: {e}")
        print("请确保已运行 'dataset_prep.py' 并生成所需的 .npy 文件。")
        return

    # 加载 LabelEncoders
    try:
        le_pitch = load_pickle('le_pitch.pkl')
        le_duration = load_pickle('le_duration.pkl')
        le_hand = load_pickle('le_hand.pkl')
        le_fingering = load_pickle('le_fingering.pkl')
    except FileNotFoundError as e:
        print(f"Error loading LabelEncoders: {e}")
        print("请确保已运行 'dataset_prep.py' 并生成相关的 .pkl 文件。")
        return

    # 加载 DataFrame
    try:
        df = pd.read_pickle('df.pkl')
        print(type(df))
    except FileNotFoundError:
        print("Error: 'df.pkl' not found. 请在 'dataset_prep.py' 中添加保存 DataFrame 的代码。")
        return

    # 计算额外特征
    df = calculate_midi_diff(df)
    df = calculate_speed_features(df)

    # 添加黑键标识符
    df['black_key'] = df['midi_number'].apply(is_black_key)
    if 'is_chord' not in df.columns:
        df['is_chord'] = 0
    df['chord'] = df['is_chord']  # 0 或 1

    # 特征提取
    feature_columns = ['pitch_encoded', 'duration_encoded', 'hand_encoded',
                       'midi_diff_processed', 'real_duration',
                       'note_density', 'black_key', 'chord']
    df = create_word_column(df, feature_columns)

    print("部分 'word' 列样例：")
    print(df['word'].head())
    save_pickle(df, "df.pkl")

    # 将 'word' 列拆分为单词列表
    tokenized_sentences = df['word'].apply(lambda x: x.split()).tolist()
    # 训练 Word2Vec-CBOW 模型
    word2vec_model = train_word2vec(tokenized_sentences, window=2, vector_size=128, min_count=1, workers=4)

    # 保存模型
    word2vec_model.save("word2vec_cbow.model")

    print("Word2Vec 模型已训练并保存。")

    # 应用CRF特征提取
    df = extract_crf_features(df, sequence_length=10)

    # 获取融合特征
    df = get_fused_features(df, word2vec_model, tokenized_sentences)

    # 将融合特征向量转化为多维特征
    fused_features = np.vstack(df['fused_feature'].values)

    # 标准化融合特征
    scaler_fused = StandardScaler()
    fused_features_scaled = scaler_fused.fit_transform(fused_features)

    # 将融合特征添加到原始特征中
    df['fused_feature_scaled'] = list(fused_features_scaled)

    # 更新特征集
    feature_columns_extended = ['pitch_encoded', 'duration_encoded', 'hand_encoded',
                               'midi_diff_processed', 'real_duration',
                               'note_density', 'black_key', 'chord']

    # 将融合特征和CRF特征附加到原始特征
    df = combine_features(df, feature_columns_extended)

    print("部分 'combined_features' 样例：")
    print(df['combined_features'].head())

    # 提取特征和标签
    X = np.stack(df['combined_features'].values)
    y = df['fingering_encoded'].values

    print(f"新特征形状: {X.shape}")
    print(f"标签形状: {y.shape}")

    # 标准化数值特征（包括融合特征）
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    # 保存标准化器
    save_pickle(scaler, 'scaler.pkl')

    # 重新创建序列
    sequence_length = 10  # 使用前10个音符预测第11个音符

    def create_sequences_full(X, y, seq_length):
        X_seq = []
        y_seq = []
        for i in range(len(X) - seq_length):
            X_seq.append(X[i:i + seq_length])
            y_seq.append(y[i + seq_length])
        return np.array(X_seq), np.array(y_seq)

    X_seq, y_seq = create_sequences_full(X, y, sequence_length)

    print(f"序列特征形状（包含融合特征）: {X_seq.shape}")  # (样本数, sequence_length, 特征数量)
    print(f"序列标签形状: {y_seq.shape}")  # (样本数,)

    # 划分训练集和验证集
    X_train_seq, X_val_seq, y_train_seq, y_val_seq = train_test_split(
        X_seq, y_seq, test_size=0.2, random_state=42, stratify=y_seq
    )

    print(f"训练集样本数: {X_train_seq.shape[0]}")
    print(f"验证集样本数: {X_val_seq.shape[0]}")

    # 移除SMOTE过采样代码
    # 直接使用原始的训练集和验证集
    X_train_resampled = X_train_seq
    y_train_resampled = y_train_seq
    X_val_resampled = X_val_seq
    y_val_resampled = y_val_seq

    print(f"处理后训练集序列形状: {X_train_resampled.shape}, 标签形状: {y_train_resampled.shape}")
    print(f"处理后验证集序列形状: {X_val_resampled.shape}, 标签形状: {y_val_resampled.shape}")

    # 数据增强：镜像对称
    def augment_mirror_symmetry(X, y, le_fingering, le_hand):
        """
        利用左右手镜像对称进行数据增强。
        """
        X_aug = []
        y_aug = []

        for i in range(len(X)):
            # 假设 'hand_encoded' 是特征中的一个维度，且为特定索引
            hand_index = 2  # 根据实际特征顺序调整
            if X[i, -1, hand_index] == le_hand.transform(['left'])[0]:
                # 将左手数据转换为右手数据
                X_mirror = X[i].copy()
                X_mirror[:, hand_index] = le_hand.transform(['right'])[0]

                # 翻转指法（具体翻转规则需根据手指编号定义）
                # 假设有 5 个手指，翻转规则如：1↔5, 2↔4, 3不变
                finger_flip = {0: 4, 1: 3, 2: 2, 3: 1, 4: 0}
                y_mirror = finger_flip.get(y[i], y[i])

                X_aug.append(X_mirror)
                y_aug.append(y_mirror)

        if X_aug:
            X_aug = np.array(X_aug)
            y_aug = np.array(y_aug)
            return np.concatenate((X, X_aug), axis=0), np.concatenate((y, y_aug), axis=0)
        else:
            return X, y

    # 执行数据增强
    X_train_aug, y_train_aug = augment_mirror_symmetry(X_train_resampled, y_train_resampled, le_fingering, le_hand)
    X_val_aug, y_val_aug = augment_mirror_symmetry(X_val_resampled, y_val_resampled, le_fingering, le_hand)

    print(f"增强后训练集序列形状: {X_train_aug.shape}, 标签形状: {y_train_aug.shape}")
    print(f"增强后验证集序列形状: {X_val_aug.shape}, 标签形状: {y_val_aug.shape}")

    # 将增强后的数据保存为 npy 文件
    np.save('X_train_aug.npy', X_train_aug)
    np.save('X_val_aug.npy', X_val_aug)
    np.save('y_train_aug.npy', y_train_aug)
    np.save('y_val_aug.npy', y_val_aug)

    print("增强后的数据已保存。")


if __name__ == "__main__":
    main()


In [ ]:
# model_training.py

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pickle
import os
from torch.utils.data import DataLoader, WeightedRandomSampler
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingWarmRestarts
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from data_utils import load_pickle
from models import TransformerModel,BiGRU,BiLSTM,BiLSTMWithAttention,CNNWithAttention,EnhancedFingeringModel

# 定义改进的 Focal Loss
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2, reduction='mean', label_smoothing=0.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
        self.label_smoothing = label_smoothing  # 添加标签平滑

    def forward(self, inputs, targets):
        # 应用标签平滑
        if self.label_smoothing > 0:
            num_classes = inputs.size(-1)
            smooth_targets = torch.zeros_like(inputs).scatter_(
                1, targets.unsqueeze(1), 1.0
            )
            smooth_targets = smooth_targets * (1 - self.label_smoothing) + self.label_smoothing / num_classes
            BCE_loss = -torch.sum(smooth_targets * F.log_softmax(inputs, dim=1), dim=1)
        else:
            BCE_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')

        pt = torch.exp(-BCE_loss)
        F_loss = (1 - pt) ** self.gamma * BCE_loss

        if self.reduction == 'mean':
            return F_loss.mean()
        elif self.reduction == 'sum':
            return F_loss.sum()
        else:
            return F_loss

# 定义 Dataset
class FingeringDataset(torch.utils.data.Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)  # 输入特征
        self.y = torch.tensor(y, dtype=torch.long)  # 指法标签

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# 评估模型性能
def evaluate_model(model, data_loader, criterion, device):
    model.eval()
    val_loss = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for X_batch, y_batch in tqdm(data_loader, desc="Evaluation", leave=False):
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            val_loss += loss.item() * X_batch.size(0)

            _, preds = torch.max(outputs, 1)
            all_labels.extend(y_batch.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    val_loss /= len(data_loader.dataset)

    # 计算总体准确率
    correct = sum(1 for p, t in zip(all_preds, all_labels) if p == t)
    accuracy = correct / len(all_labels)

    # 计算每个类别的准确率
    conf_matrix = confusion_matrix(all_labels, all_preds)
    class_accuracies = conf_matrix.diagonal() / conf_matrix.sum(axis=1)

    return val_loss, accuracy, class_accuracies, conf_matrix, all_preds, all_labels

# 修改 train_epoch 函数以支持 OneCycleLR 调度器
def train_epoch(model, train_loader, optimizer, criterion, device, clip_value=1.0):
    model.train()
    train_loss = 0
    all_labels = []
    all_preds = []
    grad_norms = []

    for X_batch, y_batch in tqdm(train_loader, desc="Training", leave=False):
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # 前向传播
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        # 反向传播
        loss.backward()

        # 计算梯度范数（用于监控梯度）
        total_norm = 0
        for p in model.parameters():
            if p.grad is not None:
                param_norm = p.grad.detach().data.norm(2)
                total_norm += param_norm.item() ** 2
        total_norm = total_norm ** 0.5
        grad_norms.append(total_norm)

        # 梯度裁剪防止梯度爆炸
        if clip_value > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_value)

        optimizer.step()

        # 更新学习率 - 批次级别更新
        if isinstance(optimizer.param_groups[0]['lr'], torch.optim.lr_scheduler.OneCycleLR):
            scheduler.step()

        train_loss += loss.item() * X_batch.size(0)

        # 记录预测结果
        _, preds = torch.max(outputs, 1)
        all_labels.extend(y_batch.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

        # 计算当前批次每个类别的准确率
        batch_labels = y_batch.cpu().numpy()
        batch_preds = preds.cpu().numpy()

        # 每100个批次检查一次类别分布
        if len(all_preds) % (100 * X_batch.size(0)) == 0:
            # 计算目前为止的类别分布
            pred_counts = np.bincount(all_preds[-1000:] if len(all_preds) > 1000 else all_preds,
                                      minlength=len(np.unique(all_labels)))

            # 如果预测严重偏向某些类别（大于70%），则警告
            max_pred_class = np.argmax(pred_counts)
            max_pred_ratio = pred_counts[max_pred_class] / pred_counts.sum()

            print(f"当前预测类别分布: {pred_counts}, 梯度范数: {np.mean(grad_norms):.4f}")

            if max_pred_ratio > 0.7:
                print(f"警告: 检测到模式崩溃 - 类别 {max_pred_class} 占比 {max_pred_ratio:.4f}")

                # 检查损失值是否异常
                if not np.isfinite(loss.item()):
                    print(f"警告: 损失值异常 ({loss.item()})")

                # 检查梯度是否异常
                if np.mean(grad_norms) > 10 or np.mean(grad_norms) < 1e-6:
                    print(f"警告: 梯度范数异常 ({np.mean(grad_norms):.4f})")

            # 重置梯度范数列表避免内存过大
            grad_norms = []

    train_loss /= len(train_loader.dataset)

    # 计算训练准确率
    correct = sum(1 for p, t in zip(all_preds, all_labels) if p == t)
    accuracy = correct / len(all_labels)

    # 计算每个类别的准确率
    class_accs = []
    unique_classes = np.unique(all_labels)
    for cls in unique_classes:
        cls_indices = [i for i, l in enumerate(all_labels) if l == cls]
        if cls_indices:
            cls_correct = sum(1 for i in cls_indices if all_preds[i] == all_labels[i])
            cls_acc = cls_correct / len(cls_indices)
            class_accs.append((cls, cls_acc))

    # 排序并输出每个类别的准确率
    class_accs.sort(key=lambda x: x[0])
    for cls, acc in class_accs:
        print(f"  类别 {cls} 训练准确率: {acc:.4f}")

    return train_loss, accuracy, all_preds, all_labels

# 可视化训练历史
def plot_training_history(history, save_path='results'):
    plt.figure(figsize=(15, 10))

    # 绘制损失曲线
    plt.subplot(2, 2, 1)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Validation Loss')
    plt.title('Loss Over Time')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    # 绘制准确率曲线
    plt.subplot(2, 2, 2)
    plt.plot(history['train_acc'], label='Train Accuracy')
    plt.plot(history['val_acc'], label='Validation Accuracy')
    plt.title('Accuracy Over Time')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    # 绘制学习率曲线
    plt.subplot(2, 2, 3)
    plt.plot(history['learning_rates'])
    plt.title('Learning Rate Over Time')
    plt.xlabel('Epoch')
    plt.ylabel('Learning Rate')

    # 绘制每个类别的最终准确率
    plt.subplot(2, 2, 4)
    class_accs = history['class_accs'][-1]
    plt.bar(range(len(class_accs)), class_accs)
    plt.title('Per-Class Accuracy (Final)')
    plt.xlabel('Class')
    plt.ylabel('Accuracy')
    plt.xticks(range(len(class_accs)))

    plt.tight_layout()
    os.makedirs(save_path, exist_ok=True)
    plt.savefig(f'{save_path}/training_history.png')
    plt.close()

# 绘制混淆矩阵
def plot_confusion_matrix(conf_matrix, save_path='results'):
    plt.figure(figsize=(10, 8))
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')

    os.makedirs(save_path, exist_ok=True)
    plt.savefig(f'{save_path}/confusion_matrix.png')
    plt.close()

def main():
    # 设置随机种子确保结果可重复
    torch.manual_seed(42)
    np.random.seed(42)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42)

    # 创建结果目录
    results_dir = 'results_cnn'
    os.makedirs(results_dir, exist_ok=True)

    # 加载数据
    try:
        # 尝试加载增强后的数据
        X_train = np.load('X_train_aug.npy')
        y_train = np.load('y_train_aug.npy')
        X_val = np.load('X_val_aug.npy')
        y_val = np.load('y_val_aug.npy')
    except FileNotFoundError:
        try:
            # 尝试加载常规训练数据
            X_train = np.load('X_train.npy')
            y_train = np.load('y_train.npy')
            X_val = np.load('X_val.npy')
            y_val = np.load('y_val.npy')
        except FileNotFoundError as e:
            print(f"无法加载数据文件: {e}")
            print("请确保已运行 'dataset_prep.py' 和 'data_process.py' 并生成所需的 .npy 文件。")
            return

    # 检查特征维度是否一致
    if X_train.shape[2] != X_val.shape[2]:
        print(f"警告: 训练集和验证集特征维度不匹配 - 训练集: {X_train.shape[2]}, 验证集: {X_val.shape[2]}")
        print("重新创建训练集和验证集，确保特征维度一致...")

        # 结合所有数据重新划分
        all_X = X_train
        all_y = y_train

        # 从训练数据重新划分
        indices = np.random.permutation(len(all_X))
        train_size = int(0.8 * len(all_X))

        train_indices = indices[:train_size]
        val_indices = indices[train_size:]

        X_train = all_X[train_indices]
        y_train = all_y[train_indices]
        X_val = all_X[val_indices]
        y_val = all_y[val_indices]

        print(f"新的训练集/验证集大小: {X_train.shape} / {X_val.shape}")

    print(f"训练集: X shape {X_train.shape}, y shape {y_train.shape}")
    print(f"验证集: X shape {X_val.shape}, y shape {y_val.shape}")

    # 输出类别分布
    unique_classes, class_counts = np.unique(y_train, return_counts=True)
    print("训练集类别分布:")
    for cls, count in zip(unique_classes, class_counts):
        print(f"  类别 {cls}: {count} 样本 ({100.0 * count / len(y_train):.2f}%)")

    # 检查数据中是否存在NaN或无穷大的值
    if np.isnan(X_train).any() or np.isinf(X_train).any():
        print("警告：训练集中存在NaN或无穷大值，将替换为0...")
        X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)

    if np.isnan(X_val).any() or np.isinf(X_val).any():
        print("警告：验证集中存在NaN或无穷大值，将替换为0...")
        X_val = np.nan_to_num(X_val, nan=0.0, posinf=0.0, neginf=0.0)

    # 使用更稳健的方式计算均值和标准差
    print("应用稳健的特征标准化...")
    X_flat_train = X_train.reshape(-1, X_train.shape[2])

    # 计算均值和标准差，使用更稳健的方法处理离群值
    q25 = np.percentile(X_flat_train, 25, axis=0)
    q75 = np.percentile(X_flat_train, 75, axis=0)
    iqr = q75 - q25

    # 将超出 IQR 范围 3 倍的值视为离群值并替换
    upper_bound = q75 + 3 * iqr
    lower_bound = q25 - 3 * iqr

    for i in range(X_flat_train.shape[1]):
        X_flat_train[:, i] = np.clip(X_flat_train[:, i], lower_bound[i], upper_bound[i])

    # 计算新的均值和标准差
    mean = np.mean(X_flat_train, axis=0)
    std = np.std(X_flat_train, axis=0)

    # 避免除零，确保标准差最小值
    std = np.maximum(std, 1e-6)

    # 应用标准化
    X_train = (X_train - mean) / std
    X_val = (X_val - mean) / std

    # 参数设置
    input_size = X_train.shape[2]  # 特征数量
    hidden_size = 512  # 增加隐藏层大小以适应CNN架构
    num_classes = len(np.unique(y_train))
    num_layers = 3  # 增加LSTM层数
    dropout = 0.5

    print(f"使用特征数量: {input_size}, 隐藏层大小: {hidden_size}, 类别数: {num_classes}")

    # 创建 Dataset 和 DataLoader
    batch_size = 128  # 增大batch size以适应CNN训练特性

    train_dataset = FingeringDataset(X_train, y_train)
    val_dataset = FingeringDataset(X_val, y_val)

    # 计算类别权重以处理数据不平衡
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(y_train),
        y=y_train
    )
    class_weights = torch.tensor(class_weights, dtype=torch.float)

    print("类别权重:")
    for i, weight in enumerate(class_weights):
        print(f"  类别 {i}: {weight:.4f}")

    # 创建加权采样器 - 修改为更平衡的采样方法
    sample_weights = np.ones_like(y_train, dtype=np.float32)
    for i, cls in enumerate(np.unique(y_train)):
        sample_weights[y_train == cls] = class_weights[i].item()

    # 归一化权重
    sample_weights = sample_weights / sample_weights.sum() * len(sample_weights)

    sampler = WeightedRandomSampler(
        sample_weights,
        len(sample_weights),
        replacement=True
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        sampler=sampler,
        num_workers=4,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=4,
        pin_memory=True
    )

    # Verify training batches for class distribution
    print("验证训练批次的类分布...")
    class_counts = np.zeros(num_classes, dtype=np.int32)
    for i, (_, y_batch) in enumerate(train_loader):
        for cls in range(num_classes):
            class_counts[cls] += (y_batch == cls).sum().item()
        if i >= 5:  # 只检查前5个批次
            break

    print("前5个批次的类分布:")
    for cls in range(num_classes):
        print(f"  类别 {cls}: {class_counts[cls]} 样本")

    # 选择设备
    device = torch.device("cuda" if torch.cuda.is_available() else
                          "mps" if torch.backends.mps.is_available() else "cpu")
    print(f"使用设备: {device}")

    # 初始化CNN模型
    model = EnhancedFingeringModel(
        input_size=input_size,
        hidden_size=hidden_size,
        num_classes=num_classes,
    ).to(device)

    print(model)

    # 计算模型参数数量
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"模型总参数: {total_params:,}")
    print(f"可训练参数: {trainable_params:,}")

    # 定义优化器 - 使用 AdamW 并调整参数，提高基础学习率
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=0.003,  # 提高初始学习率，从0.001增加到0.003
        weight_decay=0.0005,  # 轻微权重衰减
        betas=(0.9, 0.999),
        eps=1e-8
    )

    # 使用改进的 Focal Loss
    criterion = FocalLoss(
        alpha=class_weights.to(device),
        gamma=2.0,  # 恢复标准gamma值
        label_smoothing=0.1  # 轻微标签平滑
    )

    # 使用 One Cycle 学习率调度器，调整参数以提高初始学习率
    steps_per_epoch = len(train_loader)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=0.003,  # 提高最大学习率，从0.001增加到0.003
        steps_per_epoch=steps_per_epoch,
        epochs=30,
        pct_start=0.3,  # 用30%的时间来提高学习率
        div_factor=10,  # 降低除数因子，使初始学习率更高 (初始lr = max_lr/div_factor = 0.0003)
        final_div_factor=100,
        anneal_strategy='cos'
    )

    # 训练参数
    num_epochs = 30
    best_val_acc = 0.0
    patience = 10
    counter = 0
    best_model_state = None

    # 历史记录
    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': [],
        'class_accs': [],
        'learning_rates': [],
    }

    print(f"开始训练 {num_epochs} 个 epoch...")

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch + 1}/{num_epochs}")

        # 训练一个 epoch
        train_loss, train_acc, _, _ = train_epoch(
            model, train_loader, optimizer, criterion, device, clip_value=1.0
        )

        # 评估模型
        val_loss, val_acc, class_accs, conf_matrix, _, _ = evaluate_model(
            model, val_loader, criterion, device
        )

        # 更新学习率
        # 注意：在这里不调用scheduler.step()，因为我们在每个batch之后调用
        current_lr = optimizer.param_groups[0]['lr']

        # 更新历史记录
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['class_accs'].append(class_accs)
        history['learning_rates'].append(current_lr)

        # 显示训练结果
        print(f"训练损失: {train_loss:.4f} | 训练准确率: {train_acc:.4f}")
        print(f"验证损失: {val_loss:.4f} | 验证准确率: {val_acc:.4f}")
        print(f"当前学习率: {current_lr:.6f}")
        print("每个类别的准确率:")
        for i, acc in enumerate(class_accs):
            print(f"  类别 {i}: {acc:.4f}")

        # 检查是否为最佳模型
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict()
            counter = 0

            # 保存混淆矩阵
            plot_confusion_matrix(conf_matrix, results_dir)

            print(f"新的最佳模型! 验证准确率: {val_acc:.4f}")
        else:
            counter += 1
            print(f"早停计数器: {counter}/{patience}")

            # 早停
            if counter >= patience:
                print(f"早停! 在 epoch {epoch + 1} 停止训练。")
                break

    # 绘制训练历史
    plot_training_history(history, results_dir)

    # 加载最佳模型
    if best_model_state:
        model.load_state_dict(best_model_state)
        print(f"已加载最佳模型 (验证准确率: {best_val_acc:.4f})")

    # 最终评估
    _, final_acc, final_class_accs, final_conf_matrix, all_preds, all_labels = evaluate_model(
        model, val_loader, criterion, device
    )

    # 生成分类报告
    class_report = classification_report(all_labels, all_preds, digits=4)
    print("\n分类报告:")
    print(class_report)

    # 保存分类报告
    with open(f'{results_dir}/classification_report.txt', 'w') as f:
        f.write("钢琴指法预测模型 (CNN+Attention) - 分类报告\n")
        f.write("="*50 + "\n\n")
        f.write(f"验证准确率: {final_acc:.4f}\n\n")
        f.write("每个类别的准确率:\n")
        for i, acc in enumerate(final_class_accs):
            f.write(f"类别 {i}: {acc:.4f}\n")
        f.write("\n详细分类报告:\n")
        f.write(class_report)

    # 保存模型
    torch.save(model.state_dict(), f'{results_dir}/cnn_attention_model_best.pth')
    print(f"模型已保存到 {results_dir}/cnn_attention_model_best.pth")

if __name__ == "__main__":
    main()

In [ ]:
# predict_fingering.py

import os
import sys
import torch
import numpy as np
import pandas as pd
from music21 import converter
from gensim.models import Word2Vec
from sklearn.preprocessing import StandardScaler

from musicxml_utils import extract_all_features, apply_predicted_fingering
from crf_feature_extractor import extract_crf_features
from data_utils import (
    load_pickle,
    normalize_spelled_pitch,
    calculate_midi_diff,
    create_word_column,
    get_fused_features,
    combine_features,
    is_black_key,
    get_midi_number
)
from models import EnhancedFingeringModel, CNNWithAttention, BiLSTMWithAttention


def preprocess_score_data(score_path, model_dir='.'):
    """预处理乐谱数据，确保与训练数据格式完全一致"""
    print(f"开始处理乐谱: {score_path}")

    # 1. 加载乐谱和初始特征
    score = converter.parse(score_path)
    df = extract_all_features(score)
    print(f"初始特征数量: {df.shape}")

    # 2. 加载所有预处理器
    try:
        le_pitch = load_pickle(os.path.join(model_dir, 'le_pitch.pkl'))
        le_duration = load_pickle(os.path.join(model_dir, 'le_duration.pkl'))
        le_hand = load_pickle(os.path.join(model_dir, 'le_hand.pkl'))
        le_fingering = load_pickle(os.path.join(model_dir, 'le_fingering.pkl'))
        word2vec_model = Word2Vec.load(os.path.join(model_dir, "word2vec_cbow.model"))
        scaler = load_pickle(os.path.join(model_dir, 'scaler.pkl'))
        print("成功加载所有预处理器")
    except FileNotFoundError as e:
        print(f"加载预处理器错误: {e}")
        raise

    # 3. 基础特征处理
    print("处理基础特征...")

    # 标准化音高
    df['normalized_spelled_pitch'] = df['note'].apply(normalize_spelled_pitch)

    # 计算MIDI编号并添加black_key特征
    df['midi_number'] = df['normalized_spelled_pitch'].apply(get_midi_number)
    df['black_key'] = df['midi_number'].apply(is_black_key)

    # 计算MIDI差异和速度特征
    df = calculate_midi_diff(df)

    # 从现有训练数据中获取note_density的分布信息
    try:
        train_df = pd.read_pickle(os.path.join(model_dir, 'df.pkl'))
        max_density = train_df['note_density'].max()
        min_density = train_df['note_density'].min()
    except FileNotFoundError:
        print("警告：找不到训练数据，使用默认密度范围")
        max_density = 10
        min_density = 0

    # 计算音符密度
    def calculate_density(df, row_idx, window=1.0):
        """计算音符密度并归一化"""
        current_time = df.iloc[row_idx]['onset_time']
        end_time = current_time + window

        count = df[(df['onset_time'] >= current_time) &
                   (df['onset_time'] < end_time)].shape[0]

        # 归一化到训练集范围
        if max_density > min_density:
            normalized_count = (count - min_density) / (max_density - min_density) * 10
        else:
            normalized_count = count

        return normalized_count

    # 应用密度计算
    df['note_density'] = [calculate_density(df, i) for i in range(len(df))]

    # 确保chord特征存在
    if 'chord' not in df.columns:
        df['chord'] = 0

    # 4. 特征编码
    try:
        df['pitch_encoded'] = le_pitch.transform(df['normalized_spelled_pitch'])
        df['duration_encoded'] = le_duration.transform(df['duration'].astype(str))
        df['hand_encoded'] = le_hand.transform(df['hand'])
        print("特征编码成功")
    except ValueError as e:
        print(f"特征编码错误: {e}")

        # 处理未见过的值
        print("正在处理未见过的值...")

        # 处理未见过的音高
        for i, pitch in enumerate(df['normalized_spelled_pitch']):
            if pitch not in le_pitch.classes_:
                print(f"未见过的音高: {pitch}，替换为C4")
                df.at[i, 'normalized_spelled_pitch'] = 'C4'

        # 处理未见过的时值
        for i, duration in enumerate(df['duration'].astype(str)):
            if duration not in le_duration.classes_:
                print(f"未见过的时值: {duration}，替换为1.0")
                df.at[i, 'duration'] = '1.0'

        # 重新尝试编码
        df['pitch_encoded'] = le_pitch.transform(df['normalized_spelled_pitch'])
        df['duration_encoded'] = le_duration.transform(df['duration'].astype(str))
        df['hand_encoded'] = le_hand.transform(df['hand'])
        print("特征编码重试成功")

    # 5. 创建word列并获取Word2Vec特征
    feature_columns = [
        'pitch_encoded', 'duration_encoded', 'hand_encoded',
        'midi_diff_processed', 'real_duration', 'note_density',
        'black_key', 'chord'
    ]

    df = create_word_column(df, feature_columns)
    tokenized_sentences = df['word'].apply(lambda x: x.split()).tolist()

    # 6. 获取融合特征（Word2Vec）
    df = get_fused_features(df, word2vec_model, tokenized_sentences)

    # 确保fused_feature是numpy数组
    fused_features = np.vstack(df['fused_feature'].values)
    print(f"Fused feature dimension: {fused_features.shape[1]}")

    # 7. 添加CRF特征
    print("提取CRF特征...")

    # 创建零向量作为CRF特征占位符 - 与训练数据中的维度保持一致
    try:
        # 首先尝试加载CRF模型
        crf_model_path = os.path.join(model_dir, 'crf_model.pkl')
        crf = load_pickle(crf_model_path)
        print("成功加载CRF模型")

        # 使用CRF特征提取器
        df = extract_crf_features(df, crf_model_path=crf_model_path)
    except FileNotFoundError:
        print(f"警告: 找不到CRF模型，使用零向量替代")
        # 创建零向量占位符，确保维度正确（默认使用10类）
        df['crf_feature'] = [[0.0] * 10] * len(df)

    print("CRF特征提取完成")

    # 8. 检查特征维度是否与训练数据一致
    # 获取训练数据的特征维度
    try:
        sample_scaler_data = scaler.mean_.shape[0]
        expected_features = sample_scaler_data
        print(f"训练数据特征维度：{expected_features}")

        # 计算当前特征维度
        combined_dim = len(feature_columns) + fused_features.shape[1]
        if 'crf_feature' in df.columns:
            crf_dim = len(df['crf_feature'].iloc[0])
            combined_dim += crf_dim
        print(f"当前特征维度：{combined_dim}")

        # 检查是否匹配
        if combined_dim != expected_features:
            print(f"警告：特征维度不匹配！训练：{expected_features}，当前：{combined_dim}")

            # 调整CRF特征维度以匹配
            required_crf_dim = expected_features - len(feature_columns) - fused_features.shape[1]
            print(f"调整CRF特征维度为：{required_crf_dim}")
            if required_crf_dim > 0:
                df['crf_feature'] = [[0.0] * required_crf_dim] * len(df)
    except Exception as e:
        print(f"检查特征维度时出错：{e}")

    # 9. 组合所有特征
    print("组合所有特征...")
    # 使用修改后的combine_features函数
    combined_features = []

    for i, row in df.iterrows():
        # 获取原始特征
        orig_features = [float(row[col]) for col in feature_columns]

        # 获取融合特征
        fused_feature = row['fused_feature'] if 'fused_feature' in row else []

        # 获取CRF特征
        crf_feature = row['crf_feature'] if 'crf_feature' in row else []

        # 结合所有特征
        combined = np.concatenate([orig_features, fused_feature, crf_feature])
        combined_features.append(combined)

    df['combined_features'] = combined_features

    # 10. 提取并标准化特征
    X = np.stack(df['combined_features'].values)
    print(f"组合特征形状: {X.shape}")

    # 再次检查维度是否匹配
    if X.shape[1] != sample_scaler_data:
        raise ValueError(f"特征维度不匹配: 当前 {X.shape[1]}, 预期 {sample_scaler_data}")

    # 应用标准化
    X_scaled = scaler.transform(X)

    # 11. 创建序列
    sequence_length = 10
    X_seq = []

    # 处理边界条件
    padded_X = np.zeros((sequence_length-1 + len(X_scaled), X_scaled.shape[1]))
    padded_X[sequence_length-1:] = X_scaled

    for i in range(len(X_scaled)):
        X_seq.append(padded_X[i:i+sequence_length])

    X_seq = np.array(X_seq, dtype=np.float32)
    print(f"最终序列形状: {X_seq.shape}")

    return score, X_seq, df


def predict_fingering(score_path, model_path='./results_cnn/bilstm_cnn_attention_model_best.pth', model_dir='.'):
    """预测指法并应用到乐谱"""
    print(f"开始预测指法: {score_path}")

    # 预处理数据
    score, X_seq, df = preprocess_score_data(score_path, model_dir)

    # 确定输入特征维度
    input_size = X_seq.shape[2]
    print(f"模型输入特征维度: {input_size}")

    # 加载模型
    try:
        # 确定模型类型
        model_name = os.path.basename(model_path)

        # 首先加载模型权重以获取其维度
        state_dict = torch.load(model_path, map_location=torch.device('cpu'))

        # 检查模型的输入维度
        model_input_size = None

        # 尝试从权重中推断输入维度
        if 'lstm.weight_ih_l0' in state_dict:
            print("从LSTM权重推断输入维度...")
            lstm_weight = state_dict['lstm.weight_ih_l0']
            if lstm_weight.dim() > 1:
                model_input_size = lstm_weight.size(1)
                print(f"模型原始输入维度: {model_input_size}")

        # 检查CNN的隐藏层维度
        cnn_hidden_size = 128
        cnn_input_channels = input_size
        if 'conv_layers.0.weight' in state_dict:
            print("从CNN权重推断输入通道数...")
            cnn_weight = state_dict['conv_layers.0.weight']
            if cnn_weight.dim() > 0:
                cnn_hidden_size = cnn_weight.size(0)
                cnn_input_channels = cnn_weight.size(1)
                print(f"CNN隐藏层维度: {cnn_hidden_size}, 输入通道数: {cnn_input_channels}")
        elif 'conv1.weight' in state_dict:
            print("从CNN权重推断隐藏层维度...")
            cnn_weight = state_dict['conv1.weight']
            if cnn_weight.dim() > 0:
                cnn_hidden_size = cnn_weight.size(0)
                print(f"CNN隐藏层维度: {cnn_hidden_size}")

        # 如果输入维度不匹配，需要调整特征
        if model_input_size is not None and model_input_size != input_size:
            print(f"警告: 特征维度不匹配，模型期望 {model_input_size}，实际为 {input_size}")
            print("调整特征维度以匹配模型...")

            if model_input_size < input_size:
                # 如果模型期望更少的特征，裁剪特征
                print(f"裁剪特征从 {input_size} 到 {model_input_size}")
                X_seq = X_seq[:, :, :model_input_size]
            else:
                # 如果模型期望更多的特征，填充为零
                print(f"填充特征从 {input_size} 到 {model_input_size}")
                padding = torch.zeros((X_seq.shape[0], X_seq.shape[1], model_input_size - input_size), dtype=torch.float32)
                X_seq = torch.cat([torch.tensor(X_seq, dtype=torch.float32), padding], dim=2)

            # 更新输入尺寸
            input_size = model_input_size
            print(f"调整后的特征维度: {X_seq.shape}")

        # 根据模型名称创建适当的模型
        if 'cnn_attention' in model_name:
            print("使用CNN+Attention模型")
            model = CNNWithAttention(
                input_size=input_size,
                hidden_size=cnn_hidden_size,  # 使用从模型中推断的隐藏层大小
                num_classes=10,
                dropout=0.5
            )
        elif 'bilstm' in model_name.lower():
            print("使用BiLSTM+Attention模型")
            model = BiLSTMWithAttention(
                input_size=input_size,
                hidden_size=512,
                num_layers=3,
                num_classes=10,
                dropout=0.5,
                bidirectional=True
            )
        else:
            print("使用增强型指法模型")
            model = EnhancedFingeringModel(
                input_size=cnn_input_channels, # 使用检测到的输入通道数，而不是特征维度
                hidden_size=128,  # 从512改为128，以匹配保存的模型参数
                num_classes=10,
                num_heads=4
            )

        # 加载模型权重
        model.load_state_dict(state_dict)
        model.eval()
        print(f"成功加载模型: {model_path}")
    except Exception as e:
        print(f"加载模型失败: {e}")
        raise

    # 加载指法编码器
    fingering_encoder = load_pickle(os.path.join(model_dir, 'le_fingering.pkl'))

    # 预测指法
    predicted_fingerings = []
    confidence_scores = []

    print("开始预测...")
    with torch.no_grad():
        for i, x in enumerate(X_seq):
            x_tensor = torch.tensor(x).unsqueeze(0)

            # 如果使用的是增强型模型，需要调整输入形状以适应CNN的输入通道
            if isinstance(model, EnhancedFingeringModel):
                batch_size, seq_len, features = x_tensor.size()

                # 如果特征维度与CNN输入通道不匹配，需要调整
                if features != cnn_input_channels:
                    print(f"调整张量形状以匹配CNN输入通道数，将{features}特征压缩到{cnn_input_channels}通道")
                    # 将特征维度从128压缩为4通道，通过特征分组
                    if features > cnn_input_channels:
                        # 简单的方法是取均值收缩，将特征分成cnn_input_channels组
                        group_size = features // cnn_input_channels
                        x_reshaped = x_tensor.view(batch_size, seq_len, cnn_input_channels, group_size)
                        x_tensor = x_reshaped.mean(dim=3)
                        print(f"调整后的张量形状: {x_tensor.shape}")

            output = model(x_tensor)

            # 获取预测类别和概率
            probabilities = torch.nn.functional.softmax(output, dim=1)
            confidence, predicted = torch.max(probabilities, dim=1)

            finger_idx = predicted.item()
            confidence_val = confidence.item()

            # 转换为实际指法
            try:
                # 根据指法编码器转换为实际指法值
                actual_fingering = fingering_encoder.inverse_transform([finger_idx])[0]

                # 检查当前音符是否为和弦的一部分
                is_chord = False
                if i < len(df) and 'chord' in df.columns:
                    is_chord = df.iloc[i]['chord'] == 1

                # 如果是和弦，根据音符在和弦中的位置可能需要调整指法
                # 这里我们简单处理，确保每个和弦中的音符都有指法
                predicted_fingerings.append(actual_fingering)
                confidence_scores.append(confidence_val)
            except:
                print(f"警告: 无法转换指法索引 {finger_idx}，使用默认值1")
                predicted_fingerings.append(1)  # 使用默认值
                confidence_scores.append(confidence_val)

            # 打印进度
            if (i+1) % 100 == 0 or i+1 == len(X_seq):
                print(f"已处理 {i+1}/{len(X_seq)} 个音符")

    # 打印预测统计
    print("\n预测统计:")
    unique_fingers, counts = np.unique(predicted_fingerings, return_counts=True)
    for finger, count in zip(unique_fingers, counts):
        print(f"指法 {finger}: {count} 次 ({count/len(predicted_fingerings)*100:.2f}%)")

    print(f"\n平均预测置信度: {np.mean(confidence_scores):.4f}")

    # 优化预测的指法，修正不合理的连续相同指法
    print("优化预测结果，修正不合理的连续指法...")

    # 导入指法优化器
    from fingering_optimizer import FingeringOptimizer
    optimizer = FingeringOptimizer()

    # 创建临时 DataFrame 用于指法优化
    df_for_opt = df.copy()
    df_for_opt['finger_number'] = predicted_fingerings

    # 优化指法
    optimized_fingerings = optimizer.postprocess_predicted_fingerings(predicted_fingerings, df)

    # 打印优化后的统计
    print("\n优化后的指法统计:")
    unique_fingers, counts = np.unique(optimized_fingerings, return_counts=True)
    for finger, count in zip(unique_fingers, counts):
        print(f"指法 {finger}: {count} 次 ({count/len(optimized_fingerings)*100:.2f}%)")

    # 应用优化后的指法到乐谱
    modified_score = apply_predicted_fingering(score, optimized_fingerings)

    return modified_score, optimized_fingerings, confidence_scores


def main():
    """主函数"""
    import argparse

    parser = argparse.ArgumentParser(description='钢琴指法预测工具')
    parser.add_argument('input_file', type=str, help='输入的MusicXML文件路径')
    parser.add_argument('--output_file', type=str, help='输出的MusicXML文件路径')
    parser.add_argument('--model', type=str, default='./results_cnn/bilstm_cnn_attention_model_best.pth',
                        help='模型文件路径')
    parser.add_argument('--model_dir', type=str, default='.',
                        help='模型相关文件所在目录')

    args = parser.parse_args()

    # 确定输出文件名
    if args.output_file:
        output_path = args.output_file
    else:
        input_name = os.path.splitext(os.path.basename(args.input_file))[0]
        output_path = f"{input_name}_with_fingering.mxl"

    # 预测指法
    try:
        modified_score, fingerings, confidences = predict_fingering(
            args.input_file,
            model_path=args.model,
            model_dir=args.model_dir
        )

        # 保存结果
        modified_score.write('mxl', fp=output_path)
        print(f"指法预测完成。已保存到 {output_path}")

        # 保存指法和置信度数据
        confidence_path = f"{os.path.splitext(output_path)[0]}_confidences.csv"
        pd.DataFrame({
            'fingering': fingerings,
            'confidence': confidences
        }).to_csv(confidence_path, index=False)
        print(f"置信度数据已保存到 {confidence_path}")

    except Exception as e:
        print(f"处理过程中出错: {e}")
        import traceback
        traceback.print_exc()
        sys.exit(1)


if __name__ == "__main__":
    main()